# Team Topi: Phase 1 baseline (More than a Hack 2027)

Two stages. First reproduce the published nearest-player baseline exactly (sections 0-4), which
proves the setup is right. Then run our own model (package `topi/`), write `Topi.csv` next to
this notebook, validate it and score it (sections 5-9).

Rules to keep in mind: the public 5 minutes are only for checking the pipeline is correct. Never
tune on them, never train on them.

## 0. Setup

In [1]:
import os, sys, json, subprocess
import numpy as np
import pandas as pd

ROOT = os.path.abspath(".")                              # folder that holds this notebook
REPO = os.path.join(ROOT, "more-than-a-hack-2027")       # the cloned challenge repository
for sub in ("", "starter", "evaluation"):                # games.py, baseline.py, possession_metrics.py
    path = os.path.join(REPO, sub)
    if path not in sys.path:
        sys.path.insert(0, path)

import games
import baseline
import possession_metrics as pm

TEAM = "Topi"
SUBMISSION = os.path.join(ROOT, TEAM + ".csv")           # stays next to this notebook
CACHE_DIR = os.path.join(ROOT, "cache")
RESULTS_DIR = os.path.join(REPO, "evaluation", "results")
GT_PUBLIC = os.path.join(REPO, "evaluation", "gt", "BAR-ATH_possession_gt_public.csv")
N_FRAMES = games.GAMES[games.GAME]["n_video_frames"]     # 199319
os.makedirs(CACHE_DIR, exist_ok=True)

print("repo      :", REPO)
print("data dir  :", games.DATA_DIR)                     # defaults to <repo>/data, no env var needed
print("submission:", SUBMISSION)

repo      : C:\Users\user\Desktop\futbol_hackathon\more-than-a-hack-2027
data dir  : C:\Users\user\Desktop\futbol_hackathon\more-than-a-hack-2027\data
submission: C:\Users\user\Desktop\futbol_hackathon\Topi.csv


## 1. Check the data is where the code expects it

Only checks that the files exist. Nothing is parsed here.

In [2]:
for name, path in sorted(games.asset_paths(source="tactical").items()):
    size = os.path.getsize(path) / 1e6 if os.path.isfile(path) else None
    print("%-14s %-44s %s" % (name, os.path.basename(path),
                              "ok  %.1f MB" % size if size is not None else "MISSING"))

assert os.path.isfile(GT_PUBLIC), "public ground truth missing, re-clone the repo"
print()
print("graded halves (half, first video frame, last video frame):", games.half_video_blocks(games.GAME))

atd_meta       BAR-ATH_tactical_atd.xml                     ok  0.1 MB
atd_raw        BAR-ATH_tactical_atd.txt                     ok  195.7 MB
homography     BAR-ATH_tactical_homography.csv              ok  23.2 MB
smart_tagging  BAR-ATH_tactical_smart_tagging.xml           ok  0.3 MB
video          BAR-ATH_tactical.mp4                         ok  5222.3 MB

graded halves (half, first video frame, last video frame): [(1, 11185, 90651), (2, 110808, 190197)]


### 1b. Frame alignment check

One streaming pass over the raw tracking file. It must report one record per video frame,
numbered from 1, so `video_frame = frame_id - 1`.

In [3]:
import load_atd

report = load_atd.check_alignment_raw()
for key, value in report.items():
    print("%-20s %s" % (key, value))
assert report["agrees"], "ATD is not one record per video frame from 1, stop and investigate"

n_records            199319
n_video_frames       199319
frame_id_range       (1, 199319)
contiguous           True
first_video_frame    0
agrees               True


## 2. Parse the tracking once

For every frame where the ball is tracked, the team of the nearest tracked player. `-1` means
no answer (ball not tracked). Takes a minute or two, so the result is cached.

In [4]:
CODES_PATH = os.path.join(CACHE_DIR, "nearest_team_codes.npy")

if os.path.isfile(CODES_PATH):
    codes = np.load(CODES_PATH)
    print("loaded cached codes", codes.shape)
else:
    codes = baseline.build()                             # 0 = home, 1 = away, -1 = no answer
    np.save(CODES_PATH, codes)
    print("parsed and cached", codes.shape)

assert len(codes) == N_FRAMES
seen = codes != baseline.NOBODY
print("ball located on %d of %d frames (%.1f%%)" % (seen.sum(), N_FRAMES, 100 * seen.mean()))

loaded cached codes (199319,)
ball located on 113488 of 199319 frames (56.9%)


In [5]:
# Share of graded frames with a ball (docs say about 59.3%).
graded = np.zeros(N_FRAMES, dtype=bool)
for half, first, last in games.half_video_blocks(games.GAME):
    graded[first:last + 1] = True
print("graded frames: %d   ball located on graded frames: %.1f%%" % (
    graded.sum(), 100 * seen[graded].mean()))

graded frames: 158857   ball located on graded frames: 59.3%


## 3. Reproduce the four published baselines

Same algorithm four times, only the smoothing window changes (+/-0, 25, 50, 125 frames). Our
labels must match the committed `baseline_*.csv` files exactly. If they do, the setup is correct.

In [6]:
rows = []
for name, window in baseline.VARIANTS:
    ours = baseline.smooth(codes, window)
    theirs = pm.read_label_csv(os.path.join(RESULTS_DIR, name + ".csv"))["label"].astype(str).values
    same = bool(np.array_equal(ours, theirs))
    rows.append({"variant": name, "window": window, "matches_published_csv": same,
                 "n_diff_frames": int((ours != theirs).sum())})
pd.DataFrame(rows)

,variant,window,matches_published_csv,n_diff_frames
0,baseline_raw,0,True,0
1,baseline_smooth25,25,True,0
2,baseline_smooth50,50,True,0
3,baseline_smooth125,125,True,0


## 4. Score the four on the public 5 minutes

A pipeline check only (17 possessions cannot rank anything). Expected from the README:

| baseline | macroF1 | spellF1 | poss |
|---|---|---|---|
| smooth125 | 0.5812 | 0.4242 | 16 |
| smooth50 | 0.6207 | 0.3256 | 26 |
| smooth25 | 0.6200 | 0.2868 | 36 |
| raw | 0.5972 | 0.0378 | 131 |

In [7]:
gt = pm.label_codes(pm.read_label_csv(GT_PUBLIC))

def score(labels, bootstrap_n=0):
    r = pm.evaluate(gt, pm.label_codes(labels), bootstrap_n=bootstrap_n)
    return r

rows = []
for name, window in baseline.VARIANTS:
    r = score(baseline.smooth(codes, window))
    rows.append({"variant": name, "macroF1": round(r["macro_f1"], 4),
                 "spellF1": round(r["spell_f1_mean"], 4),
                 "poss": r["n_pred_spells"], "dur_s": round(r["mean_spell_duration_s"], 1)})
public_table = pd.DataFrame(rows)
print("ground truth: %d possessions, mean %.1f s" % (r["n_gt_spells"], r["gt_mean_spell_duration_s"]))
public_table

ground truth: 17 possessions, mean 14.2 s


,variant,macroF1,spellF1,poss,dur_s
0,baseline_raw,0.5972,0.0378,131,2.0
1,baseline_smooth25,0.6200,0.2868,36,7.6
2,baseline_smooth50,0.6207,0.3256,26,10.8
3,baseline_smooth125,0.5812,0.4242,16,18.5


## 5. Our model: a three-state HMM over H / A / D

Code is in `topi/` (`data.py`, `features.py`, `model.py`, `run.py`). In short:

- **Observations** per 0.2 s step, all from the tactical tracking: what the ball track is doing
  (missing, out of the pitch, stationary, moving), how fast the players are moving, the fastest
  ball movement within 2 s, and which team is closer to the ball and by how much. A ball flying
  past a player does not count as his.
- **States**: home has it, away has it, dead ball. Decoded with Viterbi, so possessions come out
  as intervals and not as flickering frames.
- **Learned without labels** (EM on the match): player speeds and recent ball speeds for dead
  versus alive, and how often play stops and restarts.
- **Fixed from football reasoning**: how much "nearest player" counts as control, one change of
  possession per ~20 s of live play, and the ball-track tables.
- Frames outside the two halves are `X`. Before a half's kick-off tag the ball is forced dead.
- Once the ball has gone out it stays dead until it is played back in at speed. Stoppages
  shorter than 3 s are removed.
- A change of possession is placed at the first moment the new team is on the ball, so a loose
  ball stays with the last team. A spell under 3 s that ends with the ball going dead (a
  clearance into touch) is not a possession.

No ground truth is used anywhere in the model, public or private.

In [8]:
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
from topi import run as topi_run, model as topi_model

labels, params = topi_run.predict()

EM log-likelihood: -140490.8 -> -138269.5 over 40 iterations
learned: stop rate 0.0080 / step, restart rate 0.0086 / step (0.2 s steps)
in-play dead-ball share: 0.473 (not calibrated)
graded frames: H 32.3%  A 19.5%  D 48.3%


In [9]:
np.set_printoptions(precision=3, suppress=True)
print("player-speed bins, slow -> fast")
print("  P(bin | alive):", params["speed_alive"])
print("  P(bin | dead) :", params["speed_dead"])
print("fastest ball movement within 2 s, slow -> fast")
print("  P(bin | alive):", params["bspeed_alive"])
print("  P(bin | dead) :", params["bspeed_dead"])
print("ball track: missing, out, still (long), still (brief), moving, (broadcast only: unused)")
print("  P(cat | alive):", params["ball_alive"])
print("  P(cat | dead) :", params["ball_dead"])

player-speed bins, slow -> fast
  P(bin | alive): [0.001 0.032 0.072 0.085 0.134 0.206 0.234 0.235]
  P(bin | dead) : [0.26  0.225 0.181 0.17  0.113 0.037 0.007 0.006]
fastest ball movement within 2 s, slow -> fast
  P(bin | alive): [0.049 0.169 0.196 0.205 0.205 0.175]
  P(bin | dead) : [0.53  0.161 0.075 0.049 0.047 0.139]
ball track: missing, out, still (long), still (brief), moving, (broadcast only: unused)
  P(cat | alive): [0.198 0.013 0.081 0.235 0.474 0.   ]
  P(cat | dead) : [0.491 0.118 0.111 0.104 0.176 0.   ]


## 6. Write `Topi.csv`

In [10]:
topi_run.write_submission(SUBMISSION, labels)
print("wrote", SUBMISSION)
print(pd.Series(labels).value_counts().to_string())

wrote C:\Users\user\Desktop\futbol_hackathon\Topi.csv
D    76662
H    51275
X    40462
A    30920


## 7. Validate the exact file

Same checks the official scorer runs: header, 199,319 rows, frame 0..199318 in order, labels in
H/A/D/X, no blanks.

In [11]:
out = subprocess.run([sys.executable, os.path.join(REPO, "evaluation", "validate_submission.py"), SUBMISSION],
                     capture_output=True, text=True)
print(out.stdout)
print(out.stderr)
assert out.returncode == 0, "validation failed, do not submit"

OK    Topi.csv
      199319 rows, frame 0..199318

      label counts
        H    51275   25.7%
        A    30920   15.5%
        D    76662   38.5%
        X    40462   20.3%

      reports 249 possessions of mean 13.2 s
      (counted with each X as a gap; the scorer bridges abstentions inside a real
       possession, so its count can be lower than this one)

Valid. Note this checks the format only -- it says nothing about whether the labels are right.
Score it against the public ground truth with run_evaluation.py.




## 8. Score on the public window, next to the baseline

A correctness check with 17 possessions. The bootstrap interval shows how wide the noise is.

In [12]:
topi_codes = pm.label_codes(pm.read_label_csv(SUBMISSION))
rows = []
for name, lab in (("baseline_smooth50 (published floor)", baseline.smooth(codes, 50)), (TEAM, None)):
    r = pm.evaluate(gt, topi_codes if lab is None else pm.label_codes(lab), bootstrap_n=1000)
    rows.append({"submission": name, "macroF1": round(r["macro_f1"], 4),
                 "spellF1": round(r["spell_f1_mean"], 4),
                 "spellF1 5-95%": "%.3f .. %.3f" % (r["spell_bootstrap_low"], r["spell_bootstrap_high"]),
                 "poss": r["n_pred_spells"], "dur_s": round(r["mean_spell_duration_s"], 1),
                 "F1 H": round(r["per_class"]["H"]["f1"], 3), "F1 A": round(r["per_class"]["A"]["f1"], 3),
                 "F1 D": round(r["per_class"]["D"]["f1"], 3),
                 "alive/dead F1": round(r["alive_dead_f1"], 3),
                 "team acc": round(r["team_acc_both_alive"], 3)})
print("ground truth: %d possessions, mean %.1f s" % (r["n_gt_spells"], r["gt_mean_spell_duration_s"]))
pd.DataFrame(rows)

ground truth: 17 possessions, mean 14.2 s


,submission,macroF1,spellF1,spellF1 5-95%,poss,dur_s,F1 H,F1 A,F1 D,alive/dead F1,team acc
0,baseline_smooth50 (published floor),0.6207,0.3256,0.214 .. 0.447,26,10.8,0.778,0.762,0.323,0.899,0.858
1,Topi,0.8157,0.5576,0.388 .. 0.715,16,16.1,0.879,0.834,0.734,0.946,0.909


In [13]:
# Whole-match sanity numbers (no ground truth needed).
runs = [(first, last + 1) for _, first, last in games.half_video_blocks(games.GAME)]
for name, lab in (("baseline_smooth50", pm.label_codes(baseline.smooth(codes, 50))), (TEAM, topi_codes)):
    s = pm.possession_stats(lab, runs)
    dead = np.mean(lab[graded] == pm.D)
    print("%-18s possessions %4d   mean %.1f s   home share %.3f   dead share %.3f" % (
        name, s["n_spells"], s["mean_duration_s"], s["home_share"], dead))

baseline_smooth50  possessions  594   mean 7.2 s   home share 0.571   dead share 0.323


Topi               possessions  249   mean 13.2 s   home share 0.624   dead share 0.483


## 9. Checklist

- [ ] section 3: all four baseline variants match the published CSVs
- [ ] section 4: public baseline scores equal the README table
- [ ] section 7: `Topi.csv` validates
- [ ] section 8: `Topi.csv` is above the baseline on both metrics, with a plausible possession count

Regenerate outside the notebook with `python -m topi.run`. Deliverables before Nov 17:
`Topi.csv`, the public repository (see `README.md`), and the technical document (`Topi.md`).